# Simple Dataset Detoxify Scoring

This notebook runs **Detoxify scoring** for a simple binary text dataset with:

- one ID column
- one text column
- one binary label column
- separate **train** and **test** files

The output schema is aligned with the unified format used by the other scoring notebooks:

- `dataset_name`
- `split`
- `detoxify_model`
- `sample_id`
- `text`
- `text_field`
- `label`
- `detox_toxicity`
- `detox_severe_toxicity`
- `detox_obscene`
- `detox_threat`
- `detox_insult`
- `detox_identity_attack`

You only need to edit the **CONFIG** cell and then run the notebook top to bottom.


In [ ]:
# Optional: install dependencies if needed
!pip install detoxify tqdm

## 1. Configuration

Edit the paths and column names below.


In [ ]:
# connect to google drive
import os
from google.colab import drive
drive.mount('/content/drive')
os.listdir('/content/drive')
os.chdir("/content/drive/My Drive/TMU-trustworthyLLM/detoxify2/")
os.getcwd()

Mounted at /content/drive


'/content/drive/My Drive/TMU-trustworthyLLM/detoxify2'

In [ ]:
CONFIG = {
    # Required
    "train_path": "hateRADAR-es/hateRADAR-es_train.csv",
    "test_path": "hateRADAR-es/hateRADAR-es_test.csv",
    "dataset_name": "hateRADAR",
    "out_root": "hateRADAR-es_detoxify_scores",

    # Column names in the raw input files
    "id_col": "id_tweet",
    "text_col": "text",
    "label_col": "label",

    # Value to write into the unified output column `text_field`
    "text_field_name": "text",

    # Which Detoxify models to run
    "variants": ["original", "unbiased", "multilingual"],

    # Inference settings
    "batch_size": 256,
    "device": None,  # auto choose cuda if available else cpu

    # Preprocessing options
    "drop_missing_label": False,
    "keep_extra_columns": False,
}
CONFIG


{'train_path': 'hateRADAR-es/hateRADAR-es_train.csv',
 'test_path': 'hateRADAR-es/hateRADAR-es_test.csv',
 'dataset_name': 'hateRADAR',
 'out_root': 'hateRADAR-es_detoxify_scores',
 'id_col': 'id_tweet',
 'text_col': 'text',
 'label_col': 'label',
 'text_field_name': 'text',
 'variants': ['original', 'unbiased', 'multilingual'],
 'batch_size': 256,
 'device': None,
 'drop_missing_label': False,
 'keep_extra_columns': False}

## 2. Imports and constants

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from detoxify import Detoxify

DETOX_VARIANTS = ["original", "unbiased", "multilingual"]
DETOX_DIMS = [
    "toxicity",
    "severe_toxicity",
    "obscene",
    "threat",
    "insult",
    "identity_attack",
]
SUPPORTED_EXTS = {".parquet", ".csv", ".tsv", ".txt"}

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


torch: 2.10.0+cu128
cuda available: True


## 3. Helper functions

In [ ]:

def detect_sep(path: Path) -> str:
    if path.suffix.lower() == ".tsv":
        return "\t"
    return ","


def read_table(path: str) -> pd.DataFrame:
    p = Path(path)
    ext = p.suffix.lower()
    if ext not in SUPPORTED_EXTS:
        raise ValueError(f"Unsupported file type: {ext}. Supported: {sorted(SUPPORTED_EXTS)}")

    if ext == ".parquet":
        return pd.read_parquet(p)
    if ext in {".csv", ".tsv", ".txt"}:
        return pd.read_csv(p, sep=detect_sep(p))
    raise ValueError(f"Unsupported file type: {ext}")


def normalize_binary_label_series(series: pd.Series) -> pd.Series:
    """
    Map common binary encodings to Int64 values {0,1}.

    Supported examples:
      0/1, "0"/"1", True/False, "true"/"false",
      safe/unsafe, benign/toxic, non-toxic/toxic, clean/toxic
    Unrecognized rows become <NA>.
    """
    raw = series.copy()

    numeric = pd.to_numeric(raw, errors="coerce")
    out = pd.Series(pd.NA, index=series.index, dtype="Int64")
    out.loc[numeric == 0] = 0
    out.loc[numeric == 1] = 1

    s = raw.astype("string").str.strip().str.lower()
    false_like = {
        "0",
        "false",
        "f",
        "no",
        "n",
        "safe",
        "benign",
        "non-toxic",
        "nontoxic",
        "clean",
        "normal",
    }
    true_like = {
        "1",
        "true",
        "t",
        "yes",
        "y",
        "unsafe",
        "toxic",
        "harmful",
        "abusive",
    }

    out.loc[s.isin(false_like)] = 0
    out.loc[s.isin(true_like)] = 1
    return out


def print_label_distribution(df: pd.DataFrame, label_col: str, prefix: str) -> None:
    print(f"\n{prefix}")
    print(f"  N = {len(df):,}")
    if label_col not in df.columns:
        print(f"  Missing label column: {label_col}")
        return

    print("  Raw value_counts:")
    print(df[label_col].value_counts(dropna=False).sort_index())

    label_bin = normalize_binary_label_series(df[label_col])
    print("  Normalized binary label counts:")
    print(label_bin.value_counts(dropna=False).sort_index())

    valid = label_bin.notna().sum()
    if valid > 0:
        neg = int((label_bin == 0).sum())
        pos = int((label_bin == 1).sum())
        print(f"  label=0: {neg:,} ({neg / valid * 100:.2f}%)")
        print(f"  label=1: {pos:,} ({pos / valid * 100:.2f}%)")
        missing = len(label_bin) - valid
        if missing > 0:
            print(f"  unmapped labels: {missing:,}")


def ensure_required_columns(df: pd.DataFrame, id_col: str, text_col: str, label_col: str, split: str) -> None:
    missing = [c for c in [id_col, text_col, label_col] if c not in df.columns]
    if missing:
        raise KeyError(
            f"[{split}] Missing required columns: {missing}. Available columns: {list(df.columns)}"
        )


def build_base_dataframe(
    df: pd.DataFrame,
    *,
    dataset_name: str,
    split: str,
    id_col: str,
    text_col: str,
    label_col: str,
    text_field_name: str,
    drop_missing_label: bool,
    keep_extra_columns: bool,
) -> pd.DataFrame:
    ensure_required_columns(df, id_col=id_col, text_col=text_col, label_col=label_col, split=split)

    label_norm = normalize_binary_label_series(df[label_col])

    base = pd.DataFrame(
        {
            "dataset_name": dataset_name,
            "split": split,
            "sample_id": df[id_col].copy(),
            "text": df[text_col].fillna("").astype(str),
            "text_field": text_field_name,
            "label": label_norm,
        },
        index=df.index,
    )

    if keep_extra_columns:
        extra = df.copy()
        rename_map = {}
        for col in extra.columns:
            if col in {"dataset_name", "split", "sample_id", "text", "text_field", "label"}:
                rename_map[col] = f"source_{col}"
        if rename_map:
            extra = extra.rename(columns=rename_map)
        base = pd.concat([base, extra], axis=1)

    if drop_missing_label:
        before = len(base)
        base = base[base["label"].notna()].copy()
        dropped = before - len(base)
        if dropped > 0:
            print(f"[{split}] Dropped {dropped:,} rows with unmapped/missing labels")

    if "label" in base.columns:
        base["label"] = base["label"].astype("Int64")

    return base


def detoxify_predict_batched(
    model: Detoxify,
    texts: List[str],
    batch_size: int,
) -> Dict[str, List[float]]:
    if len(texts) == 0:
        return {k: [] for k in DETOX_DIMS}

    output: Optional[Dict[str, List[float]]] = None
    for i in tqdm(range(0, len(texts), batch_size), desc="Detoxify scoring", leave=False):
        batch = texts[i : i + batch_size]
        scores = model.predict(batch)
        if output is None:
            output = {k: list(v) for k, v in scores.items()}
        else:
            for k in output:
                output[k].extend(scores[k])

    assert output is not None
    return output


def sanitize_for_filename(s: str) -> str:
    keep = []
    for ch in s:
        if ch.isalnum() or ch in {"-", "_"}:
            keep.append(ch)
        else:
            keep.append("-")
    out = "".join(keep).strip("-")
    while "--" in out:
        out = out.replace("--", "-")
    return out or "dataset"


def make_out_dir(out_root: str, dataset_name: str, split: str, text_field_name: str) -> str:
    dataset_slug = sanitize_for_filename(dataset_name)
    text_slug = sanitize_for_filename(text_field_name)
    out_dir = os.path.join(out_root, f"{dataset_slug}_detoxify_scores_{split}_{text_slug}")
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def run_split_scoring(
    *,
    input_path: str,
    split: str,
    dataset_name: str,
    out_root: str,
    id_col: str,
    text_col: str,
    label_col: str,
    text_field_name: str,
    variants: Iterable[str],
    batch_size: int,
    device: Optional[str],
    drop_missing_label: bool,
    keep_extra_columns: bool,
) -> List[str]:
    print(f"\n{'=' * 100}")
    print(f"Loading split='{split}' from: {input_path}")
    df = read_table(input_path)
    print(f"[{split}] Shape: {df.shape}")
    print(f"[{split}] Columns: {list(df.columns)}")
    print_label_distribution(df, label_col=label_col, prefix=f"[{dataset_name} | {split}] label distribution")

    base = build_base_dataframe(
        df,
        dataset_name=dataset_name,
        split=split,
        id_col=id_col,
        text_col=text_col,
        label_col=label_col,
        text_field_name=text_field_name,
        drop_missing_label=drop_missing_label,
        keep_extra_columns=keep_extra_columns,
    )

    out_dir = make_out_dir(out_root=out_root, dataset_name=dataset_name, split=split, text_field_name=text_field_name)
    texts = base["text"].tolist()
    n = len(texts)
    print(f"[{split}] Rows to score: {n:,}")

    if n == 0:
        print(f"[{split}] No rows left after preprocessing; skipping.")
        return []

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[{split}] Using device: {device}")

    saved_paths: List[str] = []
    dataset_slug = sanitize_for_filename(dataset_name)
    text_slug = sanitize_for_filename(text_field_name)

    unified_cols = [
        "dataset_name",
        "split",
        "detoxify_model",
        "sample_id",
        "text",
        "text_field",
        "label",
        *[f"detox_{d}" for d in DETOX_DIMS],
    ]

    for variant in variants:
        print(f"  - Running Detoxify variant: {variant}")
        model = Detoxify(variant, device=device)
        scores = detoxify_predict_batched(model, texts, batch_size=batch_size)

        missing_dims = [d for d in DETOX_DIMS if d not in scores]
        if missing_dims:
            raise KeyError(
                f"Detoxify output missing dims {missing_dims}. Available keys: {list(scores.keys())}"
            )

        out_df = base.copy()
        out_df["detoxify_model"] = variant
        for d in DETOX_DIMS:
            out_df[f"detox_{d}"] = np.asarray(scores[d], dtype=float)

        if keep_extra_columns:
            ordered = unified_cols + [c for c in out_df.columns if c not in unified_cols]
            out_df = out_df[ordered]
        else:
            out_df = out_df[unified_cols]

        fname = f"scores__{dataset_slug}__{split}__{text_slug}__detoxify-{variant}__N{n}.parquet"
        fpath = os.path.join(out_dir, fname)
        out_df.to_parquet(fpath, index=False)
        saved_paths.append(fpath)
        print(f"    [Saved] {fpath}")

    return saved_paths


def run_full_pipeline(config: dict) -> List[str]:
    required = ["train_path", "test_path", "dataset_name", "out_root", "id_col", "text_col", "label_col", "text_field_name"]
    missing = [k for k in required if k not in config]
    if missing:
        raise KeyError(f"Missing required CONFIG keys: {missing}")

    variants = config.get("variants", DETOX_VARIANTS)
    bad = [v for v in variants if v not in DETOX_VARIANTS]
    if bad:
        raise ValueError(f"Unsupported variants: {bad}. Allowed: {DETOX_VARIANTS}")

    print("Configuration:")
    for k, v in config.items():
        print(f"  {k}: {v}")

    train_paths = run_split_scoring(
        input_path=config["train_path"],
        split="train",
        dataset_name=config["dataset_name"],
        out_root=config["out_root"],
        id_col=config["id_col"],
        text_col=config["text_col"],
        label_col=config["label_col"],
        text_field_name=config["text_field_name"],
        variants=variants,
        batch_size=config.get("batch_size", 256),
        device=config.get("device", None),
        drop_missing_label=config.get("drop_missing_label", False),
        keep_extra_columns=config.get("keep_extra_columns", False),
    )

    test_paths = run_split_scoring(
        input_path=config["test_path"],
        split="test",
        dataset_name=config["dataset_name"],
        out_root=config["out_root"],
        id_col=config["id_col"],
        text_col=config["text_col"],
        label_col=config["label_col"],
        text_field_name=config["text_field_name"],
        variants=variants,
        batch_size=config.get("batch_size", 256),
        device=config.get("device", None),
        drop_missing_label=config.get("drop_missing_label", False),
        keep_extra_columns=config.get("keep_extra_columns", False),
    )

    saved_paths = train_paths + test_paths

    print("\nDone.")
    print("Saved files:")
    for p in saved_paths:
        print(" -", p)

    return saved_paths


## 4. Optional sanity check

This cell loads the raw train/test files and prints column names plus label distribution before scoring.


In [ ]:
train_df_preview = read_table(CONFIG["train_path"])
test_df_preview = read_table(CONFIG["test_path"])

print("Train shape:", train_df_preview.shape)
print("Train columns:", list(train_df_preview.columns))
print_label_distribution(train_df_preview, CONFIG["label_col"], "[TRAIN]")

print("\n" + "-" * 100 + "\n")

print("Test shape:", test_df_preview.shape)
print("Test columns:", list(test_df_preview.columns))
print_label_distribution(test_df_preview, CONFIG["label_col"], "[TEST]")


Train shape: (4000, 3)
Train columns: ['id_tweet', 'text', 'label']

[TRAIN]
  N = 4,000
  Raw value_counts:
label
0    3048
1     952
Name: count, dtype: int64
  Normalized binary label counts:
0    3048
1     952
Name: count, dtype: Int64
  label=0: 3,048 (76.20%)
  label=1: 952 (23.80%)

----------------------------------------------------------------------------------------------------

Test shape: (1000, 3)
Test columns: ['id_tweet', 'text', 'label']

[TEST]
  N = 1,000
  Raw value_counts:
label
0    762
1    238
Name: count, dtype: int64
  Normalized binary label counts:
0    762
1    238
Name: count, dtype: Int64
  label=0: 762 (76.20%)
  label=1: 238 (23.80%)


## 5. Run Detoxify scoring

Run this cell to generate the unified parquet outputs.


In [ ]:
saved_paths = run_full_pipeline(CONFIG)
saved_paths


Configuration:
  train_path: hateRADAR-es/hateRADAR-es_train.csv
  test_path: hateRADAR-es/hateRADAR-es_test.csv
  dataset_name: hateRADAR
  out_root: hateRADAR-es_detoxify_scores
  id_col: id_tweet
  text_col: text
  label_col: label
  text_field_name: text
  variants: ['original', 'unbiased', 'multilingual']
  batch_size: 256
  device: None
  drop_missing_label: False
  keep_extra_columns: False

Loading split='train' from: hateRADAR-es/hateRADAR-es_train.csv
[train] Shape: (4000, 3)
[train] Columns: ['id_tweet', 'text', 'label']

[hateRADAR | train] label distribution
  N = 4,000
  Raw value_counts:
label
0    3048
1     952
Name: count, dtype: int64
  Normalized binary label counts:
0    3048
1     952
Name: count, dtype: Int64
  label=0: 3,048 (76.20%)
  label=1: 952 (23.80%)
[train] Rows to score: 4,000
[train] Using device: cuda
  - Running Detoxify variant: original
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to

100%|██████████| 418M/418M [00:11<00:00, 36.8MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

... (32 outputs left out)

XLMRobertaForSequenceClassification LOAD REPORT from: None
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Detoxify scoring:   0%|          | 0/4 [00:00<?, ?it/s]

    [Saved] hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_test_text/scores__hateRADAR__test__text__detoxify-multilingual__N1000.parquet

Done.
Saved files:
 - hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_train_text/scores__hateRADAR__train__text__detoxify-original__N4000.parquet
 - hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_train_text/scores__hateRADAR__train__text__detoxify-unbiased__N4000.parquet
 - hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_train_text/scores__hateRADAR__train__text__detoxify-multilingual__N4000.parquet
 - hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_test_text/scores__hateRADAR__test__text__detoxify-original__N1000.parquet
 - hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_test_text/scores__hateRADAR__test__text__detoxify-unbiased__N1000.parquet
 - hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_test_text/scores__hateRADAR__test__text__detoxify-multilingual__N1000.parquet


['hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_train_text/scores__hateRADAR__train__text__detoxify-original__N4000.parquet',
 'hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_train_text/scores__hateRADAR__train__text__detoxify-unbiased__N4000.parquet',
 'hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_train_text/scores__hateRADAR__train__text__detoxify-multilingual__N4000.parquet',
 'hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_test_text/scores__hateRADAR__test__text__detoxify-original__N1000.parquet',
 'hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_test_text/scores__hateRADAR__test__text__detoxify-unbiased__N1000.parquet',
 'hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_test_text/scores__hateRADAR__test__text__detoxify-multilingual__N1000.parquet']

## 6. Inspect one saved parquet

Optional quick check of one output file.


In [ ]:
if saved_paths:
    sample_out = saved_paths[0]
    out_df = pd.read_parquet(sample_out)
    print(sample_out)
    display(out_df.head())
    print(out_df.dtypes)


hateRADAR-es_detoxify_scores/hateRADAR_detoxify_scores_train_text/scores__hateRADAR__train__text__detoxify-original__N4000.parquet


,dataset_name,split,detoxify_model,sample_id,text,text_field,label,detox_toxicity,detox_severe_toxicity,detox_obscene,detox_threat,detox_insult,detox_identity_attack
0,hateRADAR,train,original,707301374057390080,Ojalá algún día se juzgue por crímenes contra ...,text,0,0.048105,0.000320,0.002725,0.000695,0.001195,0.001256
1,hateRADAR,train,original,682609731375083520,Desaparecida 9 años. Teléfono ANAR para Casos ...,text,0,0.010100,0.000118,0.000759,0.000194,0.000438,0.000252
2,hateRADAR,train,original,712378523814666240,@user opinar. Si estuviera en Londres y tuvier...,text,1,0.026081,0.000148,0.001642,0.000274,0.000792,0.000347
3,hateRADAR,train,original,794242711146659840,#Entérate Operación militar en #Mosul deja más...,text,0,0.001409,0.000097,0.000213,0.000118,0.000190,0.000145
4,hateRADAR,train,original,731769029489229824,Ayer en #PINEO #Renedo tuvieron debates sobre ...,text,0,0.256608,0.000469,0.006582,0.000991,0.003662,0.003965


dataset_name              object
split                     object
detoxify_model            object
sample_id                  int64
text                      object
text_field                object
label                      Int64
detox_toxicity           float64
detox_severe_toxicity    float64
detox_obscene            float64
detox_threat             float64
detox_insult             float64
detox_identity_attack    float64
dtype: object
